# Spain Power System — 2035 NECPEssentials Market Chain

Results of the operational market chain (DA → ID2 → ID3 → CID → Balancing → AC OPF redispatch)
run on the **OpenEMPIRE NECPEssentials 2035 portfolio** (period 2, 2030–2035), scaled onto the
2024 Spanish network. Same figure style as `results_plots.ipynb` (2024 case).

**Three parts:**
- **Part I** — the 2035 dispatch, generation mix, branch loading and bus voltages.
- **Part II** — *Feasibility conclusions*: the AC OPF redispatch is infeasible on the un-reinforced
  2024 grid; here we quantify how much intra-Spain reinforcement the 2035 fleet needs.
- **Part III** — *System adequacy*: is demand always served (any load shedding)? Firm-capacity margin,
  duration curves and winter scarcity.

> **Data provenance.** The redispatch CSVs in `results/NECPEssentials/` come from a feasible run with
> `line_rating_factor` raised well above the 0.80 baseline (the headroom that makes all 48 hours
> feasible — see Part II). Branch loadings below are therefore expressed against those inflated
> ratings. The exact factor is **not hardcoded here** — the setup cell reads it from `config.toml
> [network]`, and the branch-flow map recovers the *effective* factor directly from the CSVs
> (`limit_mw / nominal`), so every label tracks the data regardless of the current config. The
> reinforcement analysis then backs out the true per-corridor requirement from those flows.

In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

pio.templates.default = 'plotly_white'

# Outputs from the NECPEssentials scenario run (label = "NECPEssentials")
RESULTS = Path('results/GoRES')

# The line_rating_factor these redispatch CSVs were generated with. Read it from
# config.toml [network] so every map label/colourbar tracks the actual setting instead
# of a hardcoded constant. (Re-run the OPF after changing config so the CSVs match.)
def _read_lrf(path='config.toml'):
    try:
        import tomllib          # Python 3.11+
        loader, mode = tomllib, 'rb'
    except ModuleNotFoundError:
        import tomli as loader   # fallback for < 3.11
        mode = 'rb'
    with open(path, mode) as f:
        return float(loader.load(f)['network']['line_rating_factor'])

LRF_USED = _read_lrf()
print(f'LRF_USED (from config.toml [network]) = {LRF_USED:g}')

FUEL_COLORS = {
    'Wind':                '#4CAF50',
    'Solar':               '#FFC107',
    'Hydro':               '#2196F3',
    'Nuclear':             '#9C27B0',
    'Gas':                 '#F44336',
    'Coal':                '#212121',
    'Biomass':             '#795548',
    'Oil':                 '#607D8B',
    'CrossBorder':         '#6C757D',
    'Battery':             '#00BCD4',
    'Electricity Storage': '#00BCD4',
    'LoadShed':            '#E91E63',
    'Slack':               '#BDBDBD',
}

def save_plotly(fig, filename):
    out = RESULTS / filename
    fig.write_html(out, include_plotlyjs='cdn', full_html=True)
    return out

# Spain geo frame reused by every map
GEO = dict(projection_type='mercator', lonaxis_range=[-10, 5], lataxis_range=[35.5, 44.5],
           showland=True, landcolor='#f5f5f2', showcoastlines=True, coastlinecolor='#888888',
           showcountries=True, countrycolor='#aaaaaa', showocean=True, oceancolor='#eef5fb')
print('config ready - RESULTS =', RESULTS)

LRF_USED (from config.toml [network]) = 1
config ready - RESULTS = results\GoRES


In [2]:
# ---- load the scenario outputs ----
summary = pd.read_csv(RESULTS / 'summary.csv')
gen     = pd.read_csv(RESULTS / 'gen_dispatch.csv')
fuel    = pd.read_csv(RESULTS / 'fuel_mix.csv')
branch  = pd.read_csv(RESULTS / 'branch_flows.csv')
volt    = pd.read_csv(RESULTS / 'bus_voltages.csv')

# bus coordinates (x = lon, y = lat); file carries a UTF-8 BOM on the first header
buses = pd.read_csv('Data/Bus_Data.csv', encoding='utf-8-sig')
buses_pos = buses.reset_index(drop=True).copy()
buses_pos.index = range(1, len(buses_pos) + 1)   # branch from_bus/to_bus are 1-based row indices


def effective_lrf(branch_df, lines_df, fallback=None):
    """Recover the line_rating_factor actually baked into branch_flows.csv, so every
    label AND derived number tracks the data regardless of the current config.toml.
    The OPF limit is limit_mw = nominal × lrf and real nominal = sqrt(3)·V·Imax·circuits,
    so lrf ≈ median(limit_mw / nominal) over AC lines (transformers/DC use a different
    rating formula and are skipped). Returns `fallback` if nothing usable is found."""
    L = lines_df
    circ = pd.to_numeric(L['circuits'], errors='coerce').fillna(1.0)
    cap  = np.sqrt(3) * L['voltage'].astype(float) * L['Imax'].astype(float) * circ
    cap_by = dict(zip(L['line_id'].astype(str), cap))
    dc_col = L['dc'] if 'dc' in L.columns else pd.Series('f', index=L.index)
    dc_by  = {str(a): str(b).lower() in ('t', 'true', '1') for a, b in zip(L['line_id'], dc_col)}
    lim = branch_df.groupby('branch_name')['limit_mw'].first()
    effs = []
    for name, limit_mw in lim.items():
        lid = str(name); c = cap_by.get(lid)
        if (lid.startswith('TR_') or dc_by.get(lid)
                or not c or not np.isfinite(c) or c <= 0 or not np.isfinite(limit_mw)):
            continue
        effs.append(limit_mw / c)
    return float(np.median(effs)) if effs else (float(fallback) if fallback is not None else float('nan'))


# Canonical rating factor for THIS run: recovered from the CSVs (self-consistent), with the
# config.toml value only as a fallback. Everything below (labels + req_factor) uses LRF_USED.
_lines_for_lrf = pd.read_csv('Data/lines.csv')
LRF_CONFIG = LRF_USED                                   # what config.toml [network] currently says
LRF_USED   = effective_lrf(branch, _lines_for_lrf, fallback=LRF_CONFIG)
LRF_DISP   = round(LRF_USED, 2)
print(f'line_rating_factor recovered from CSVs = {LRF_USED:.2f}  '
      f'(config.toml [network] = {LRF_CONFIG:g})')
if np.isfinite(LRF_USED) and abs(LRF_USED - LRF_CONFIG) > 0.05:
    print(f'  NOTE: these CSVs were generated at ~{LRF_DISP:g}×, not the current config {LRF_CONFIG:g}× '
          f'— re-run the OPF if you want them to match.')

DAYS = sorted(summary['date'].unique())
DAY_LABEL = {DAYS[0]: f'{DAYS[0]} (summer)', DAYS[-1]: f'{DAYS[-1]} (winter)'}
n_solved = (summary['status'].isin(['OPTIMAL', 'LOCALLY_SOLVED'])).sum()
print(f'{len(DAYS)} days, {len(summary)} hourly redispatch rows, {n_solved} solved')
print('days:', DAYS)

line_rating_factor recovered from CSVs = 1.00  (config.toml [network] = 1)
2 days, 48 hourly redispatch rows, 48 solved
days: ['2024-07-08', '2024-12-02']


## Part I — 2035 dispatch and network state

### Hourly generation mix
Redispatched generation by fuel across the day. Note the extreme 2035 midday solar (exceeds load),
curtailed and exported, and the reservoir-hydro / wind backbone overnight.

In [3]:
fig = make_subplots(rows=1, cols=len(DAYS), shared_yaxes=True,
                    subplot_titles=[DAY_LABEL[d] for d in DAYS])
order = ['Nuclear','Hydro','Wind','Solar','Biomass','Coal','Gas','Oil','Battery','CrossBorder','LoadShed']
for ci, d in enumerate(DAYS, start=1):
    sub = fuel[fuel['date'] == d]
    piv = sub.pivot_table(index='hour', columns='fuel', values='dispatch_mw', aggfunc='sum').fillna(0)
    for f in order:
        if f not in piv.columns:
            continue
        y = piv[f].clip(lower=0) / 1000.0
        if y.abs().sum() < 1e-6:
            continue
        fig.add_trace(go.Scatter(x=piv.index, y=y, name=f, legendgroup=f,
                                 showlegend=(ci == 1), stackgroup='mix',
                                 mode='lines', line=dict(width=0.5, color=FUEL_COLORS.get(f, '#999')),
                                 fillcolor=FUEL_COLORS.get(f, '#999')), row=1, col=ci)
    ld = summary[summary['date'] == d].sort_values('hour')
    fig.add_trace(go.Scatter(x=ld['hour'], y=ld['total_load_mw'] / 1000.0, name='load',
                             legendgroup='load', showlegend=(ci == 1), mode='lines',
                             line=dict(color='#111', width=2, dash='dot')), row=1, col=ci)
    fig.update_xaxes(title_text='hour', row=1, col=ci)
fig.update_yaxes(title_text='GW', row=1, col=1)
fig.update_layout(title='2035 NECPEssentials — hourly generation mix (redispatch)',
                  height=460, hovermode='x unified')
save_plotly(fig, 'mix_2035.html'); fig.show()

### Installed capacity vs. mean dispatch, by fuel

In [4]:
cap = (fuel.groupby('fuel').agg(capacity_mw=('capacity_mw', 'max'),
                                mean_disp=('dispatch_mw', 'mean')).reset_index())
cap = cap[cap['capacity_mw'] > 1].sort_values('capacity_mw', ascending=False)
fig = go.Figure()
fig.add_bar(x=cap['fuel'], y=cap['capacity_mw'] / 1000.0, name='installed capacity',
            marker_color='#cfd8dc')
fig.add_bar(x=cap['fuel'], y=cap['mean_disp'] / 1000.0, name='mean dispatch',
            marker_color=[FUEL_COLORS.get(f, '#999') for f in cap['fuel']])
fig.update_layout(title='2035 installed capacity vs mean redispatch', barmode='overlay',
                  yaxis_title='GW', height=430, hovermode='x unified')
fig.update_traces(opacity=0.85)
save_plotly(fig, 'capacity_vs_dispatch_2035.html'); fig.show()

### Hourly system cost and load
Redispatch objective (€/h) against served load. Winter is far more expensive (low RES, scarcity);
summer midday is near-zero cost (RES-abundant).

In [5]:
fig = make_subplots(specs=[[{'secondary_y': True}]])
for d in DAYS:
    s = summary[summary['date'] == d].sort_values('hour')
    fig.add_trace(go.Scatter(x=s['hour'], y=s['objective_eur_h'] / 1000.0,
                             name=f'cost {DAY_LABEL[d]}', mode='lines+markers'), secondary_y=False)
    fig.add_trace(go.Scatter(x=s['hour'], y=s['total_load_mw'] / 1000.0,
                             name=f'load {DAY_LABEL[d]}', mode='lines', line=dict(dash='dot')),
                  secondary_y=True)
fig.update_layout(title='2035 redispatch — hourly cost and load', height=430, hovermode='x unified',
                  xaxis_title='hour')
fig.update_yaxes(title_text='cost [k€/h]', secondary_y=False)
fig.update_yaxes(title_text='load [GW]', secondary_y=True)
save_plotly(fig, 'hourly_cost_2035.html'); fig.show()

### Branch loading map — peak hour
Line loading (% of the inflated rating used in this feasible run — the exact ×-nominal is recovered
from the CSVs and shown in the title) at the highest-load snapshot.

In [6]:
def branch_segments(df, val_col, bins, colors, base_width, name_prefix):
    """Group branches into color bins, one Scattergeo line trace per bin (None-separated segments)."""
    traces = []
    for lo, hi, col in zip(bins[:-1], bins[1:], colors):
        sel = df[(df[val_col] >= lo) & (df[val_col] < hi)]
        if sel.empty:
            continue
        lons, lats = [], []
        for _, r in sel.iterrows():
            b0, b1 = buses_pos.loc[int(r['from_bus'])], buses_pos.loc[int(r['to_bus'])]
            lons += [b0['x'], b1['x'], None]
            lats += [b0['y'], b1['y'], None]
        traces.append(go.Scattergeo(lon=lons, lat=lats, mode='lines',
                                    line=dict(width=base_width, color=col),
                                    name=f'{name_prefix} {lo:g}–{hi:g}',
                                    hoverinfo='skip'))
    return traces

peak = summary.loc[summary['total_load_mw'].idxmax()]
snap = branch[(branch['date'] == peak['date']) & (branch['hour'] == int(peak['hour']))].copy()
bins   = [0, 25, 50, 75, 90, 100, 1e9]
colors = ['#1a9850', '#91cf60', '#fee08b', '#fc8d59', '#d73027', '#7a0177']
fig = go.Figure(branch_segments(snap, 'loading_pct', bins, colors, 1.6, 'load %'))
fig.update_geos(**GEO)
fig.update_layout(title=f"2035 branch loading — {peak['date']} h{int(peak['hour']):02d} "
                        f"(load {peak['total_load_mw']/1000:.1f} GW, % of {LRF_DISP:g}× rating)",
                  height=650, margin=dict(l=10, r=10, t=60, b=10),
                  legend=dict(title='loading', y=0.9))
save_plotly(fig, 'branch_loading_2035.html'); fig.show()

### Bus voltage map — mean over all hours

In [7]:
vm = volt.groupby('bus_id')['vm_pu'].agg(vm_mean='mean', vm_min='min', vm_max='max').reset_index()
g = buses.merge(vm, on='bus_id', how='left').dropna(subset=['vm_mean'])
fig = go.Figure(go.Scattergeo(
    lon=g['x'], lat=g['y'], mode='markers', text=g['bus_id'],
    marker=dict(size=7, color=g['vm_mean'], colorscale='RdYlGn', cmin=0.95, cmax=1.05, cmid=1.0,
                colorbar=dict(title='V [pu]'), line=dict(color='#333', width=0.2), opacity=0.9),
    customdata=np.column_stack([g['vm_min'], g['vm_max']]),
    hovertemplate='Bus %{text}<br>mean %{marker.color:.3f}<br>min %{customdata[0]:.3f}'
                  '<br>max %{customdata[1]:.3f}<extra></extra>'))
fig.update_geos(**GEO)
fig.update_layout(title='2035 bus voltage — mean over all redispatch hours',
                  height=650, margin=dict(l=10, r=10, t=60, b=10))
save_plotly(fig, 'bus_voltage_2035.html'); fig.show()
print('bus-hours <0.95pu: %d   >1.05pu: %d' % ((volt.vm_pu < 0.95).sum(), (volt.vm_pu > 1.05).sum()))

bus-hours <0.95pu: 0   >1.05pu: 0


## Part II — Feasibility conclusions

Full analysis in `results/NECPEssentials/feasibility_findings.md`. The market stages (DA→Balancing)
all clear, but on the **un-reinforced 2024 grid the AC OPF redispatch is infeasible on every hour**.
It is a **purely thermal** limit (voltage relaxation had zero effect), fixable with **targeted
corridor reinforcement**.

### Feasibility vs. line-rating headroom
Sweeping `line_rating_factor` (thermal headroom over nominal): 0/48 → 36/48 → 48/48.

In [8]:
# results of the diagnostic sweep (this session)
sweep = pd.DataFrame({'line_rating_factor': [0.8, 2.0, 4.0],
                      'feasible_hours': [0, 36, 48]})
fig = go.Figure(go.Bar(x=sweep['line_rating_factor'].astype(str), y=sweep['feasible_hours'],
                       text=[f'{h}/48' for h in sweep['feasible_hours']], textposition='outside',
                       marker_color=['#d73027', '#fee08b', '#1a9850']))
fig.add_hline(y=48, line=dict(color='#999', dash='dot'), annotation_text='all hours')
fig.update_layout(title='AC OPF redispatch feasibility vs line_rating_factor (×nominal)',
                  xaxis_title='line_rating_factor', yaxis_title='feasible hours (of 48)',
                  yaxis_range=[0, 52], height=420)
save_plotly(fig, 'feasibility_sweep_2035.html'); fig.show()

### The wall is at low load, not the solar peak
At `line_rating_factor = 2.0` the failing hours are the **lowest-load overnight hours** (~30 GW),
where flat ~11.6 GW northern hydro + ~10.6 GW wind export south on corridors 2× can't carry — the
midday solar peak (with local southern supply) solves.

In [9]:
# hours that were infeasible at line_rating_factor = 2.0 (from the diagnostic run)
fail_2p0 = {DAYS[0]: {1, 2, 3, 4, 5}, DAYS[-1]: {0, 1, 2, 3, 4, 5, 6}}
s = summary.copy()
s['feasible@2.0×'] = ~s.apply(lambda r: int(r['hour']) in fail_2p0.get(r['date'], set()), axis=1)
fig = go.Figure()
for d in DAYS:
    sd = s[s['date'] == d].sort_values('hour')
    ok = sd[sd['feasible@2.0×']]; bad = sd[~sd['feasible@2.0×']]
    fig.add_trace(go.Scatter(x=ok['hour'], y=ok['total_load_mw']/1000, mode='markers+lines',
                             name=f'{DAY_LABEL[d]} — feasible', line=dict(width=1),
                             marker=dict(size=8, color='#1a9850')))
    fig.add_trace(go.Scatter(x=bad['hour'], y=bad['total_load_mw']/1000, mode='markers',
                             name=f'{DAY_LABEL[d]} — infeasible @2×', marker=dict(size=11, color='#d73027', symbol='x')))
fig.add_hline(y=33, line=dict(color='#999', dash='dot'), annotation_text='~33 GW threshold')
fig.update_layout(title='Feasibility at line_rating_factor=2.0 vs hourly load',
                  xaxis_title='hour', yaxis_title='load [GW]', height=440, hovermode='x unified')
save_plotly(fig, 'feasibility_by_load_2035.html'); fig.show()

### How much reinforcement each corridor needs
For every branch, the minimum rating factor it requires = peak `loading_pct × LRF_USED / 100`
(from the feasible run's flows, with `LRF_USED` recovered from the CSVs). The need is **targeted,
not blanket**.

In [10]:
req = (branch.groupby(['branch_id', 'branch_name'])
       .agg(peak_load_pct=('loading_pct', 'max')).reset_index())
req['req_factor'] = req['peak_load_pct'] * LRF_USED / 100.0
req = req.sort_values('req_factor', ascending=False)

thresholds = [0.8, 1.0, 1.5, 2.0, 3.0]
counts = [int((req['req_factor'] > t).sum()) for t in thresholds]
print('system-min feasible line_rating_factor ~ %.2f  (branch %s)'
      % (req['req_factor'].max(), req.iloc[0]['branch_name']))
for t, c in zip(thresholds, counts):
    print(f'  branches needing > {t:>3}x : {c}')

fig = make_subplots(rows=1, cols=2, column_widths=[0.45, 0.55],
                    subplot_titles=('branches above a rating factor', 'top 12 binding corridors'),
                    specs=[[{'type': 'bar'}, {'type': 'bar'}]])
fig.add_bar(x=[f'>{t}×' for t in thresholds], y=counts, marker_color='#3987e5',
            text=counts, textposition='outside', row=1, col=1)
top = req.head(12).iloc[::-1]
fig.add_bar(x=top['req_factor'], y=top['branch_name'], orientation='h',
            marker_color='#d73027', text=top['req_factor'].round(2), textposition='outside',
            row=1, col=2)
fig.update_xaxes(title_text='min rating factor needed', row=1, col=2)
fig.update_yaxes(title_text='branches', row=1, col=1)
fig.update_layout(title='2035 intra-Spain reinforcement requirement', height=470, showlegend=False)
save_plotly(fig, 'reinforcement_bars_2035.html'); fig.show()
req.head(12)

system-min feasible line_rating_factor ~ 1.00  (branch TR_ec3a7de9)
  branches needing > 0.8x : 104
  branches needing > 1.0x : 0
  branches needing > 1.5x : 0
  branches needing > 2.0x : 0
  branches needing > 3.0x : 0


,branch_id,branch_name,peak_load_pct,req_factor
1337,2201,TR_ec3a7de9,100.0,0.999999
1066,1959,LTGES1284,100.0,0.999999
271,1242,LTGES0816,100.0,0.999999
1320,2187,TR_10cbc112,100.0,0.999999
1439,2294,TR_4cb0b32d,100.0,0.999999
166,1148,LTGES0756,100.0,0.999999
1309,2177,NEWES_ES211_ES220,100.0,0.999999
2044,721,LTGES0497,100.0,0.999999
1431,2287,TR_71f49079,100.0,0.999999
1503,2351,TR_5d89fb1a,100.0,0.999999


### Reinforcement map — where the 2024 grid must be strengthened
Branches drawn by required rating factor: grey = adequate (≤ nominal), warming colors = needs
reinforcement. The handful of red corridors (≳2×, led by LTGES0690/0691/1322) are the binding
constraints for hosting the 2035 NECPEssentials fleet.

In [11]:
import json, urllib.request

bins   = [0, 1.0, 1.5, 2.0, 3.0]
colors = ['#c9ccd1', '#fee08b', '#fc8d59', '#d73027']
widths = [0.6, 1.6, 2.4, 3.4]
merged = branch[['branch_id', 'from_bus', 'to_bus']].drop_duplicates('branch_id') \
    .merge(req[['branch_id', 'branch_name', 'req_factor']], on='branch_id', how='inner')

# ── NUTS2 borders from Eurostat GISCO (change LEVL_2 → LEVL_3 for provinces) ──
_nuts_url = ('https://gisco-services.ec.europa.eu/distribution/v2/nuts/geojson/'
             'NUTS_RG_20M_2021_4326_LEVL_3.geojson')
with urllib.request.urlopen(_nuts_url) as _r:
    _nuts = json.load(_r)
_es_feats = [f for f in _nuts['features'] if f['properties']['CNTR_CODE'] == 'ES']

fig = go.Figure()
# draw NUTS2 region borders first so they sit underneath the branch overlay
for _feat in _es_feats:
    _geom = _feat['geometry']
    _polys = _geom['coordinates'] if _geom['type'] == 'MultiPolygon' else [_geom['coordinates']]
    for _poly in _polys:
        for _ring in _poly:
            _lons = [c[0] for c in _ring] + [_ring[0][0]]
            _lats = [c[1] for c in _ring] + [_ring[0][1]]
            fig.add_trace(go.Scattergeo(lon=_lons, lat=_lats, mode='lines',
                                        line=dict(color='#888888', width=0.8),
                                        showlegend=False, hoverinfo='skip'))

for lo, hi, col, w in zip(bins[:-1], bins[1:], colors, widths):
    sel = merged[(merged['req_factor'] >= lo) & (merged['req_factor'] < hi)]
    if sel.empty:
        continue
    lons, lats = [], []
    for _, r in sel.iterrows():
        b0, b1 = buses_pos.loc[int(r['from_bus'])], buses_pos.loc[int(r['to_bus'])]
        lons += [b0['x'], b1['x'], None]; lats += [b0['y'], b1['y'], None]
    lbl = 'adequate (≤1×)' if hi == 1.0 else f'{lo:g}–{hi:g}× needed'
    fig.add_trace(go.Scattergeo(lon=lons, lat=lats, mode='lines',
                                line=dict(width=w, color=col), name=lbl, hoverinfo='skip'))
# label the top-5 binding corridors at their midpoints
top5 = merged.sort_values('req_factor', ascending=False).head(5)
tlon = [(buses_pos.loc[int(r['from_bus'])]['x'] + buses_pos.loc[int(r['to_bus'])]['x']) / 2 for _, r in top5.iterrows()]
tlat = [(buses_pos.loc[int(r['from_bus'])]['y'] + buses_pos.loc[int(r['to_bus'])]['y']) / 2 for _, r in top5.iterrows()]
fig.add_trace(go.Scattergeo(lon=tlon, lat=tlat, mode='markers+text',
    text=[f"{n}<br>{f:.2f}×" for n, f in zip(top5['branch_name'], top5['req_factor'])],
    textposition='top center', textfont=dict(size=10, color='#7a0177'),
    marker=dict(size=6, color='#7a0177'), name='top corridors'))
fig.update_geos(**GEO)
fig.update_layout(title='2035 NECPEssentials — intra-Spain reinforcement map (min rating factor needed)',
                  height=680, margin=dict(l=10, r=10, t=60, b=10), legend=dict(y=0.9))
save_plotly(fig, 'reinforcement_map_2035.html'); fig.show()


### Interactive grid map (folium)

One self-contained HTML map with an in-map control panel (top right). Saved to
`results/grid_maps/branch_flow_2035.html` and opened in the browser.

**Choose the frame** — a dropdown + slider covering all 51 frames: each of the 48 individual
hours, a 24-hour average per day, and the average over all hours. Everything is precomputed and
embedded once (~3 MB), so switching frames only restyles the existing Leaflet layers (~25 ms)
and the whole map stays a single portable file. `INITIAL_FRAME` in the cell picks which frame it
opens on; the panel reaches the rest.

**Choose the bus overlay** — radio buttons for *none* / *generation* / *demand* / *net injection* /
*voltage magnitude*.

- The three power quantities size each circle by value, on a scale fixed across frames so sizes
  stay comparable as you step through hours. Net injection is diverging: red = surplus (the bus
  exports into the grid), blue = deficit (it imports). Per-bus demand is reconstructed the way
  `run_opf.jl` assigns it — `Data/load.csv` share × hourly system load — and net injection is read
  straight off the branch flows, so the three close on each other:
  `generation + load shed − demand = net injection` (verified to <0.1 MW per bus).
- **Voltage** is the AC OPF solution from `bus_voltages.csv`. Per-unit voltage is intensive, so
  circle size would mean nothing here and colour carries it instead: a diverging blue → white →
  red scale centred on 1.0 pu, spanning whatever the run produced but never narrower than ±5 %.
  Buses outside ±5 % are flagged in the popup. This overlay only says something when
  `[redispatch].power_flow = "AC"` — a DC redispatch has no voltage solution and `run_opf.jl`
  falls back to a flat 1.0 pu, which the cell detects and reports rather than drawing a uniform
  field that looks like a result.

Every bus popup reports generation, demand, net injection, voltage magnitude and angle, any load
shed, and installed capacity by fuel — all for the selected frame.

**Lines** are coloured by loading for the selected frame and their width scales with real capacity
from `lines.csv`. Clicking one gives capacity, the OPF limit, the flow **and its direction**
(`ES00359 → ES00242`), stated in the sign convention of `branch_flows.csv` rather than the
`lines.csv` column order. On an averaged frame the popup reports the mean magnitude alongside the
net signed flow and flags corridors whose direction reverses within the period. An optional
overlay draws direction arrows on the 180 most-loaded corridors of the current frame.

The map is not rendered inline by default (`SHOW_INLINE = False`) so that ~3 MB does not get
written into the `.ipynb` on every save.

In [ ]:
# ============================================================================
# Interactive branch-flow + bus map, with an in-map frame selector.
#
#   * frame        = a single hour, a whole-day average, or the all-hours average
#   * bus overlay  = none / generation / demand / net injection / voltage
#                    (radio buttons; voltage needs [redispatch].power_flow = "AC")
#   * line popup   = flow magnitude AND direction (from_bus -> to_bus)
#
# All 51 frames are precomputed here and embedded once (~2.3 MB); the map JS only
# restyles the existing Leaflet layers, so switching frames is ~25 ms and never
# re-downloads geometry.
# ============================================================================
import json, webbrowser
import folium
from branca.colormap import LinearColormap

OUT_MAPS = Path('results/grid_maps'); OUT_MAPS.mkdir(parents=True, exist_ok=True)
lines_g  = pd.read_csv('Data/lines.csv')
load_sh  = pd.read_csv('Data/load.csv')                 # bus_id -> demand share

# Which frame the map OPENS on (every frame stays reachable from the in-map panel):
#   'peak'          -> the highest-load hour        (default)
#   'all'           -> average over every hour
#   '2024-07-08'    -> that day's 24-hour average
#   '2024-07-08 13' -> that single hour
INITIAL_FRAME = 'peak'

# Rendering the 2.3 MB map inline would bake it into the .ipynb on every save,
# so by default we only write the HTML and open it in the browser.
SHOW_INLINE = False

# ---------------------------------------------------------------------------
# 1. bus table: the 1-based row index of Bus_Data.csv is the `bus_i` / `from_bus`
#    key used by gen_dispatch.csv and branch_flows.csv.
# ---------------------------------------------------------------------------
B = buses.reset_index(drop=True).copy()
B['bus_i'] = np.arange(1, len(B) + 1)
NB = len(B)
bus_ix = pd.Index(B['bus_i'])                                   # 1..NB
id_of  = B['bus_id'].tolist()

# demand shares -> per-bus fraction of the system load (run_opf.jl does exactly this:
# pd = demand / sum(demand) * total_load_mw), so per-bus demand is exactly recoverable
share = B[['bus_id']].merge(load_sh, on='bus_id', how='left')['demand'].fillna(0.0).to_numpy(float)
share = share / share.sum()

# ---------------------------------------------------------------------------
# 2. frame index: 24 h per day, then one average per day, then the grand average
# ---------------------------------------------------------------------------
hours_idx  = summary[['date', 'hour']].drop_duplicates().sort_values(['date', 'hour'])
hour_keys  = list(map(tuple, hours_idx.to_numpy()))             # [(date, hour), ...]
NH         = len(hour_keys)
hour_pos   = {k: i for i, k in enumerate(hour_keys)}
day_rows   = {d: [i for i, (dd, _) in enumerate(hour_keys) if dd == d] for d in DAYS}

frame_labels = [f'{d} h{int(h):02d}' for d, h in hour_keys]
frame_labels += [f'{d} — day average' for d in DAYS]
frame_labels += ['all hours — average']
frame_groups = [f'{d} (hourly)' for d, _ in hour_keys] + ['averages'] * (len(DAYS) + 1)
NF = len(frame_labels)


def with_aggregates(M):
    """Stack the NH hourly rows of M with one mean row per day plus a grand mean."""
    day_means = np.vstack([M[day_rows[d]].mean(axis=0) for d in DAYS])
    return np.vstack([M, day_means, M.mean(axis=0, keepdims=True)])


# ---------------------------------------------------------------------------
# 3. per-bus, per-hour quantities
# ---------------------------------------------------------------------------
PSEUDO = {'LoadShed', 'Slack'}                                  # not physical plant
g_real = gen[~gen['fuel'].isin(PSEUDO)]
G = (g_real.pivot_table(index=['date', 'hour'], columns='bus_i', values='dispatch_mw',
                        aggfunc='sum', fill_value=0.0)
     .reindex(index=hour_keys, columns=bus_ix, fill_value=0.0).to_numpy(float))

SHED = (gen[gen['fuel'] == 'LoadShed']
        .pivot_table(index=['date', 'hour'], columns='bus_i', values='dispatch_mw',
                     aggfunc='sum', fill_value=0.0)
        .reindex(index=hour_keys, columns=bus_ix, fill_value=0.0).to_numpy(float))

tot_load = (summary.set_index(['date', 'hour'])['total_load_mw']
            .reindex(hour_keys).to_numpy(float))
D = tot_load[:, None] * share[None, :]                          # gross demand [MW]

# Net injection taken straight from the branch flows, so it is the true nodal balance
# rather than a derived quantity. It closes against the other two to <0.1 MW per bus:
#   generation + load shed - demand = net injection = net outflow on the incident lines
NET = np.zeros((NH, NB))
fb = branch['from_bus'].to_numpy() - 1
tb = branch['to_bus'].to_numpy() - 1
ft = np.array([hour_pos[(d, h)] for d, h in zip(branch['date'], branch['hour'])])
np.add.at(NET, (ft, fb),  branch['flow_mw'].to_numpy(float))
np.add.at(NET, (ft, tb), -branch['flow_mw'].to_numpy(float))

# Bus voltage magnitude and angle from the AC OPF. A DC redispatch has no voltage
# solution and run_opf.jl falls back to vm = 1.0 pu, so check for a flat field rather
# than drawing a uniform map that looks like a result.
_vcols = pd.Index(id_of)
VM = (volt.pivot_table(index=['date', 'hour'], columns='bus_id', values='vm_pu', aggfunc='mean')
      .reindex(index=hour_keys, columns=_vcols).to_numpy(float))
VA = (volt.pivot_table(index=['date', 'hour'], columns='bus_id', values='va_deg', aggfunc='mean')
      .reindex(index=hour_keys, columns=_vcols).to_numpy(float))

G_F, D_F, N_F, S_F, V_F, A_F = (with_aggregates(M) for M in (G, D, NET, SHED, VM, VA))

_vfin = V_F[np.isfinite(V_F)]
VM_LO, VM_HI = (float(_vfin.min()), float(_vfin.max())) if _vfin.size else (1.0, 1.0)
VM_FLAT = (VM_HI - VM_LO) < 1e-6
STATUTORY = 0.05                                # +/-5 % is the band worth flagging
if VM_FLAT:
    print(f'WARNING: vm_pu = {VM_LO:.4f} at every bus and hour. These CSVs come from a DC '
          f'redispatch, which carries no voltage solution — the voltage overlay will be '
          f'uniform until you re-run with [redispatch].power_flow = "AC".')
else:
    _lo = int((volt['vm_pu'] < 1 - STATUTORY).sum())
    _hi = int((volt['vm_pu'] > 1 + STATUTORY).sum())
    print(f'bus voltage: {VM_LO:.4f} .. {VM_HI:.4f} pu   '
          f'(outside +/-{STATUTORY:.0%}: {_lo} bus-hours low, {_hi} high, '
          f'of {len(volt)})')

# static per-bus context for the popup: installed capacity by fuel
_units = gen.drop_duplicates(['gen_id'])
cap_txt = {}
for bi, sub in _units[~_units['fuel'].isin(PSEUDO)].groupby('bus_i'):
    parts = sub.groupby('fuel')['capacity_mw'].sum().sort_values(ascending=False)
    cap_txt[int(bi)] = ', '.join(f'{f} {v:.0f}' for f, v in parts.items() if v >= 1)

# ---------------------------------------------------------------------------
# 4. per-line, per-hour quantities (geometry + ratings come from lines.csv)
# ---------------------------------------------------------------------------
lines_g['circuits_numeric']   = pd.to_numeric(lines_g['circuits'], errors='coerce').fillna(1.0)
lines_g['cap_mw_per_circuit'] = np.sqrt(3) * lines_g['voltage'].astype(float) * lines_g['Imax'].astype(float)
lines_g['cap_mw_real']        = lines_g['cap_mw_per_circuit'] * lines_g['circuits_numeric']

LRF_EFF  = effective_lrf(branch, lines_g, fallback=LRF_USED)
LRF_DISP = round(LRF_EFF, 2)

pos_of_bus_id = {b: i for i, b in enumerate(id_of)}              # 'ES00001' -> 0-based row
FLOW = branch.pivot_table(index=['date', 'hour'], columns='branch_name',
                          values='flow_mw', aggfunc='sum').reindex(hour_keys)
LOAD = branch.pivot_table(index=['date', 'hour'], columns='branch_name',
                          values='loading_pct', aggfunc='max').reindex(hour_keys)
ends = branch.drop_duplicates('branch_name').set_index('branch_name')[['from_bus', 'to_bus', 'limit_mw']]

geo, meta, cols_kept, n_missing = [], [], [], 0
for _, r in lines_g.iterrows():
    lid = str(r['line_id'])
    i0, i1 = pos_of_bus_id.get(r['bus0']), pos_of_bus_id.get(r['bus1'])
    if i0 is None or i1 is None:
        n_missing += 1
        continue
    has = lid in FLOW.columns and lid in ends.index
    if has:
        e = ends.loc[lid]
        # branch_flows' own from/to define the sign convention of flow_mw, which is
        # what the popup turns into a direction — do not assume the lines.csv order
        fi, ti = int(e['from_bus']) - 1, int(e['to_bus']) - 1
        limit  = round(float(e['limit_mw']), 1)
    else:
        fi, ti, limit = i0, i1, None
    geo.append([i0, i1])
    meta.append([lid, int(r['voltage']), round(float(r['cap_mw_real']), 1),
                 round(float(r['cap_mw_per_circuit']), 1), float(r['circuits_numeric']),
                 1 if str(r.get('dc', 'f')).lower() in ('t', 'true', '1') else 0,
                 limit, fi, ti])
    cols_kept.append(lid if has else None)

kept = [c for c in cols_kept if c is not None]
F_F = with_aggregates(FLOW.reindex(columns=kept).fillna(0.0).to_numpy(float))
L_F = with_aggregates(LOAD.reindex(columns=kept).fillna(0.0).to_numpy(float))
res_col, _j = {}, 0                                             # line row -> column in F_F/L_F
for i, c in enumerate(cols_kept):
    if c is not None:
        res_col[i] = _j; _j += 1

print(f'frames: {NF}  ({NH} hourly + {len(DAYS)} day averages + 1 grand average)')
print(f'lines drawn: {len(geo)}  with flow results: {len(kept)}  (skipped, bus missing: {n_missing})')
print(f'labelling line loading against {LRF_DISP:g}x nominal (recovered from the CSVs)')

# ---------------------------------------------------------------------------
# 5. base map + legend
# ---------------------------------------------------------------------------
m = folium.Map(location=[40.0, -3.6], zoom_start=6, tiles='cartodbpositron', control_scale=True)

LOAD_RAMP = ['#1a9850', '#91cf60', '#fee08b', '#fc8d59', '#d73027', '#7a0177']
vmax = float(max(100.0, np.nanmax(L_F)))
LinearColormap(LOAD_RAMP, vmin=0, vmax=vmax,
               caption=f'line loading [% of {LRF_DISP:g}x rating] — for the selected frame').add_to(m)

# NUTS-3 province borders as a toggleable layer (optional — skipped offline)
try:
    import urllib.request
    _u = ('https://gisco-services.ec.europa.eu/distribution/v2/nuts/geojson/'
          'NUTS_RG_20M_2021_4326_LEVL_3.geojson')
    with urllib.request.urlopen(_u, timeout=30) as _r:
        _nuts = json.load(_r)
    _nuts['features'] = [f for f in _nuts['features'] if f['properties']['CNTR_CODE'] == 'ES']
    fg = folium.FeatureGroup(name='NUTS-3 borders', show=False)
    folium.GeoJson(_nuts, style_function=lambda _f: dict(color='#888888', weight=0.8, fill=False)).add_to(fg)
    fg.add_to(m)
except Exception as _e:
    print('NUTS-3 layer skipped:', _e)

# ---------------------------------------------------------------------------
# 6. payload -> JS
# ---------------------------------------------------------------------------
if INITIAL_FRAME == 'peak':
    _pk = summary.loc[summary['total_load_mw'].idxmax()]
    f0 = hour_pos[(_pk['date'], int(_pk['hour']))]
elif INITIAL_FRAME == 'all':
    f0 = NF - 1
elif INITIAL_FRAME in DAYS:
    f0 = NH + DAYS.index(INITIAL_FRAME)
else:
    _d, _h = INITIAL_FRAME.split()
    f0 = hour_pos[(_d, int(_h))]

CAP_MIN, CAP_MAX = float(lines_g['cap_mw_real'].min()), float(lines_g['cap_mw_real'].max())
widths = [round(0.8 + 5.2 * (mm[2] - CAP_MIN) / max(CAP_MAX - CAP_MIN, 1e-9), 2) for mm in meta]

payload = dict(
    busLL=[[round(float(y), 5), round(float(x), 5)] for y, x in zip(B['y'], B['x'])],
    busId=id_of,
    busKv=[int(v) for v in B['voltage']],
    busCap=[cap_txt.get(int(i), '') for i in B['bus_i']],
    lineGeo=geo,
    lineMeta=[mm + [w] for mm, w in zip(meta, widths)],
    lineCol=[res_col.get(i, -1) for i in range(len(geo))],
    F=[[round(float(v), 1) for v in row] for row in F_F],
    L=[[round(float(v), 1) for v in row] for row in L_F],
    G=[[round(float(v), 1) for v in row] for row in G_F],
    D=[[round(float(v), 1) for v in row] for row in D_F],
    N=[[round(float(v), 1) for v in row] for row in N_F],
    V=[[(None if not np.isfinite(v) else round(float(v), 4)) for v in row] for row in V_F],
    A=[[(None if not np.isfinite(v) else round(float(v), 1)) for v in row] for row in A_F],
    S=[[[j, round(float(v), 1)] for j, v in enumerate(row) if abs(v) > 0.05] for row in S_F],
    labels=frame_labels, groups=frame_groups, f0=f0,
    ramp=LOAD_RAMP, vmax=vmax, lrf=LRF_DISP,
    vmFlat=bool(VM_FLAT), vmLo=VM_LO, vmHi=VM_HI, vmBand=STATUTORY,
)
blob = json.dumps(payload, separators=(',', ':'))
print(f'embedded payload: {len(blob) / 1e6:.1f} MB')

CSS = """
<style>
.gridctl{background:#fff;padding:8px 10px;border-radius:6px;box-shadow:0 1px 5px rgba(0,0,0,.4);
         font:12px/1.35 -apple-system,Segoe UI,Helvetica,Arial,sans-serif;min-width:216px;max-width:250px}
.gridctl b{font-size:12px}
.gridctl select,.gridctl input[type=range]{width:100%;margin:3px 0;font-size:12px}
.gridctl label{display:block;cursor:pointer;padding:1px 0}
.gridctl hr{border:0;border-top:1px solid #ddd;margin:7px 0}
.gridctl .lbl{font-weight:600;color:#1565c0;margin:2px 0 4px}
.gridctl .hint{color:#777;font-size:10.5px;margin-top:4px}
.gridpop{font:12px/1.4 -apple-system,Segoe UI,Helvetica,Arial,sans-serif}
.gridpop .dir{background:#eef5fb;padding:2px 5px;border-radius:3px;display:inline-block;margin:2px 0}
</style>
"""
m.get_root().header.add_child(folium.Element(CSS))

JS = """
// folium emits this block into <head>, i.e. before the map object exists, so defer
// everything until the page (and the map's own inline script) has finished loading.
window.addEventListener('load', function(){
var MAP = __MAP__;
var P = __DATA__;
var frame = P.f0, busMode = 'none';

// ---- colour helpers -------------------------------------------------------
function hex2rgb(h){return [parseInt(h.substr(1,2),16),parseInt(h.substr(3,2),16),parseInt(h.substr(5,2),16)];}
function lerpRamp(ramp, t){
  t = Math.max(0, Math.min(1, t)) * (ramp.length - 1);
  var i = Math.min(ramp.length - 2, Math.floor(t)), f = t - i, a = ramp[i], b = ramp[i+1];
  return 'rgb(' + Math.round(a[0]+(b[0]-a[0])*f) + ',' + Math.round(a[1]+(b[1]-a[1])*f)
       + ',' + Math.round(a[2]+(b[2]-a[2])*f) + ')';
}
var RAMP = P.ramp.map(hex2rgb);
function rampColor(v){ return lerpRamp(RAMP, v / P.vmax); }

// voltage: diverging about 1.0 pu, spanning whatever the OPF actually produced but
// never narrower than the +/-5 % band, so a compliant run still reads as compliant
var VRAMP = ['#2166ac','#67a9cf','#f7f7f7','#ef8a62','#b2182b'].map(hex2rgb);
var VDEV = Math.max(P.vmBand, Math.abs(P.vmHi - 1), Math.abs(1 - P.vmLo));
function vmColor(v){
  return (v === null || v === undefined) ? '#bbbbbb'
       : lerpRamp(VRAMP, (v - (1 - VDEV)) / (2 * VDEV));
}
function fmt(v, d){
  if (v === null || v === undefined || !isFinite(v)) return 'n/a';
  // group the integer part only — grouping the fraction would render 0.03 as "0.0 30"
  var s = v.toFixed(d === undefined ? 0 : d).split('.');
  s[0] = s[0].replace(/\\B(?=(\\d{3})+(?!\\d))/g, ' ');
  return s.join('.');
}

// ---- load shed is sparse, so it ships as [busIndex, MW] pairs per frame ----
var SHED = P.S.map(function(rows){ var o = {}; rows.forEach(function(p){ o[p[0]] = p[1]; }); return o; });

// ---- lines ----------------------------------------------------------------
var lineGroup = L.layerGroup().addTo(MAP), lineLayers = [];
P.lineGeo.forEach(function(g, i){
  var md = P.lineMeta[i];
  var pl = L.polyline([P.busLL[g[0]], P.busLL[g[1]]],
                      {color:'#cccccc', weight:md[9], opacity:0.85,
                       dashArray: md[5] ? '5,6' : null, interactive:true});
  // popups/tooltips are built at interaction time so they always show the current frame
  pl.bindTooltip(function(){ return lineTip(i); }, {sticky:true});
  pl.on('click', function(e){ pl.bindPopup(linePopup(i), {maxWidth:340}).openPopup(e.latlng); });
  pl.addTo(lineGroup);
  lineLayers.push(pl);
});

function lineState(i){
  var c = P.lineCol[i];
  if (c < 0) return null;
  var md = P.lineMeta[i], f = P.F[frame][c], lp = P.L[frame][c], lim = md[6];
  // on an averaged frame f is the mean SIGNED flow (net direction) while lp is the
  // mean loading, so |flow| comes from the loading and can exceed |f| when reversing
  var absf = (lim && isFinite(lim)) ? lp / 100 * lim : Math.abs(f);
  var a = P.busId[md[7]], b = P.busId[md[8]];
  return {f:f, lp:lp, lim:lim, absf:absf,
          from: f >= 0 ? a : b, to: f >= 0 ? b : a,
          reversing: absf > Math.abs(f) * 1.001 + 0.5};
}
function lineTip(i){
  var md = P.lineMeta[i], s = lineState(i);
  if (!s) return '<b>' + md[0] + '</b><br>' + md[1] + ' kV — no flow result';
  return '<b>' + md[0] + '</b><br>' + fmt(s.absf) + ' MW (' + s.lp.toFixed(0) + ' %)'
       + '<br>' + s.from + ' &rarr; ' + s.to;
}
function linePopup(i){
  var md = P.lineMeta[i], s = lineState(i);
  var h = '<div class="gridpop"><b>' + md[0] + '</b> | ' + md[1] + ' kV | circuits ' + md[4]
        + '<br><b>real capacity (lines.csv): ' + fmt(md[2]) + ' MW</b>'
        + '<br>per-circuit rating: ' + fmt(md[3]) + ' MW';
  if (!s) return h + '<br><i>(no flow result)</i></div>';
  h += '<br>OPF limit (' + P.lrf + 'x nominal): ' + fmt(s.lim) + ' MW'
     + '<hr style="margin:5px 0"><b>' + P.labels[frame] + '</b>'
     + '<br><span class="dir">' + s.from + ' &rarr; ' + s.to + '</span>'
     + '<br>flow: <b>' + fmt(s.absf) + ' MW</b> (' + s.lp.toFixed(1) + ' % of limit)';
  if (s.reversing)
    h += '<br>net over the period: ' + fmt(Math.abs(s.f)) + ' MW in that direction'
       + '<br><i>(flow reverses direction within the period)</i>';
  h += '<br><span style="color:#777">endpoints in flow convention: ' + P.busId[md[7]]
     + ' &rarr; ' + P.busId[md[8]] + ' (positive = this way)</span></div>';
  return h;
}

// ---- flow-direction arrows (top-N lines of the current frame) -------------
var ARROW_N = 180;
var arrowGroup = L.layerGroup();
function bearing(a, b){
  var y1 = a[0]*Math.PI/180, y2 = b[0]*Math.PI/180, dx = (b[1]-a[1])*Math.PI/180;
  var y = Math.sin(dx)*Math.cos(y2);
  var x = Math.cos(y1)*Math.sin(y2) - Math.sin(y1)*Math.cos(y2)*Math.cos(dx);
  return (Math.atan2(y, x)*180/Math.PI + 360) % 360;
}
function rebuildArrows(){
  arrowGroup.clearLayers();
  var rank = [];
  for (var i = 0; i < P.lineGeo.length; i++){
    var s = lineState(i);
    if (s && s.absf > 1) rank.push([s.absf, i, s]);
  }
  rank.sort(function(p, q){ return q[0] - p[0]; });
  rank.slice(0, ARROW_N).forEach(function(e){
    var i = e[1], s = e[2], g = P.lineGeo[i];
    var a = P.busLL[g[0]], b = P.busLL[g[1]];
    // geometry endpoints follow lines.csv; flip if the flow convention disagrees
    var geoFwd = P.busId[g[0]] === s.from;
    var p1 = geoFwd ? a : b, p2 = geoFwd ? b : a;
    var mid = [(a[0]+b[0])/2, (a[1]+b[1])/2];
    var deg = bearing(p1, p2);
    L.marker(mid, {interactive:false, icon: L.divIcon({className:'gridarrow',
      iconSize:[16,16], iconAnchor:[8,8],
      html:'<div style="transform:rotate(' + (deg - 90) + 'deg);color:'
         + rampColor(s.lp) + ';font-size:15px;line-height:16px;text-align:center;'
         + 'text-shadow:0 0 2px #fff,0 0 2px #fff">&#10148;</div>'})}).addTo(arrowGroup);
  });
}

// ---- buses ----------------------------------------------------------------
var busGroup = L.layerGroup(), busLayers = [];
var BUSMAX = {gen:1, dem:1, net:1};
['G','D','N'].forEach(function(k, idx){
  var key = ['gen','dem','net'][idx], mx = 1;
  P[k].forEach(function(row){ row.forEach(function(v){ var a = Math.abs(v); if (a > mx) mx = a; }); });
  BUSMAX[key] = mx;                       // fixed across frames, so sizes stay comparable
});
var BUSCOL = {gen:'#2e7d32', dem:'#ef6c00', pos:'#c62828', neg:'#1565c0'};

P.busLL.forEach(function(ll, j){
  var cm = L.circleMarker(ll, {radius:0, weight:0.6, color:'#333', opacity:0.55,
                               fillOpacity:0.6, fillColor:'#999', interactive:true});
  cm.bindTooltip(function(){ return busTip(j); }, {sticky:true});
  cm.on('click', function(e){ cm.bindPopup(busPopup(j), {maxWidth:320}).openPopup(e.latlng); });
  cm.addTo(busGroup);
  busLayers.push(cm);
});

function busValue(j){
  if (busMode === 'gen') return P.G[frame][j];
  if (busMode === 'dem') return P.D[frame][j];
  if (busMode === 'net') return P.N[frame][j];
  if (busMode === 'vm')  return P.V[frame][j];
  return 0;
}
var MODE_NAME = {gen:'generation', dem:'demand', net:'net injection',
                 vm:'voltage magnitude'};
function busTip(j){
  if (busMode === 'none') return '<b>' + P.busId[j] + '</b><br>' + P.busKv[j] + ' kV';
  var v = busValue(j);
  if (busMode === 'vm')
    return '<b>' + P.busId[j] + '</b><br>voltage: '
         + (v === null || v === undefined ? 'n/a' : v.toFixed(4) + ' pu');
  var extra = busMode === 'net' ? ' (' + (v >= 0 ? 'surplus, exports' : 'deficit, imports') + ')' : '';
  return '<b>' + P.busId[j] + '</b><br>' + MODE_NAME[busMode] + ': ' + fmt(v, 1) + ' MW' + extra;
}
function busPopup(j){
  var g = P.G[frame][j], d = P.D[frame][j], n = P.N[frame][j], sh = SHED[frame][j] || 0;
  var h = '<div class="gridpop"><b>' + P.busId[j] + '</b> | ' + P.busKv[j] + ' kV'
        + '<hr style="margin:5px 0"><b>' + P.labels[frame] + '</b>'
        + '<br>generation: <b>' + fmt(g, 1) + ' MW</b>'
        + '<br>demand: <b>' + fmt(d, 1) + ' MW</b>'
        + '<br>net injection: <b>' + fmt(n, 1) + ' MW</b> ('
        + (n >= 0 ? 'surplus &rarr; exports to the grid' : 'deficit &larr; imports from the grid') + ')';
  var vm = P.V[frame][j], va = P.A[frame][j];
  if (vm !== null && vm !== undefined){
    var off = Math.abs(vm - 1) > P.vmBand + 1e-9;
    h += '<br>voltage: <b' + (off ? ' style="color:#c62828"' : '') + '>' + vm.toFixed(4) + ' pu</b>'
       + (va === null || va === undefined ? '' : ', angle ' + va.toFixed(1) + '&deg;')
       + (P.vmFlat ? ' <span style="color:#c62828">(DC run — no voltage solution)</span>'
                   : (off ? ' <span style="color:#c62828">(outside &plusmn;'
                          + (P.vmBand * 100).toFixed(0) + ' %)</span>' : ''));
  }
  if (sh > 0.05) h += '<br><span style="color:#c62828">load shed: ' + fmt(sh, 1) + ' MW</span>';
  if (P.busCap[j]) h += '<br><span style="color:#777">installed [MW]: ' + P.busCap[j] + '</span>';
  return h + '</div>';
}

// ---- redraw ---------------------------------------------------------------
function redraw(){
  for (var i = 0; i < lineLayers.length; i++){
    var s = lineState(i);
    lineLayers[i].setStyle({color: s ? rampColor(s.lp) : '#cccccc'});
  }
  if (MAP.hasLayer(arrowGroup)) rebuildArrows();
  if (busMode === 'none'){
    if (MAP.hasLayer(busGroup)) MAP.removeLayer(busGroup);
  } else {
    if (!MAP.hasLayer(busGroup)) busGroup.addTo(MAP);
    var mx = BUSMAX[busMode];
    for (var j = 0; j < busLayers.length; j++){
      var v = busValue(j), r, col;
      if (busMode === 'vm'){
        // pu is intensive — a bigger circle would mean nothing, so colour carries it
        r   = (v === null || v === undefined) ? 0 : 4.5;
        col = vmColor(v);
      } else {
        var a = Math.abs(v);
        r   = a < 0.5 ? 0 : 2 + 12 * Math.sqrt(a / mx);
        col = busMode === 'net' ? (v >= 0 ? BUSCOL.pos : BUSCOL.neg) : BUSCOL[busMode];
      }
      busLayers[j].setStyle({radius:r, fillColor:col, color:'#333',
                             weight: r > 0 ? 0.6 : 0, fillOpacity: 0.65});
    }
  }
  var el = document.getElementById('gridFrameLbl');
  if (el) el.textContent = P.labels[frame];
  var lg = document.getElementById('gridBusLegend');
  if (lg) lg.innerHTML = legendHtml();
}
function legendHtml(){
  if (busMode === 'none') return '<span class="hint">No bus overlay.</span>';
  if (busMode === 'vm'){
    if (P.vmFlat)
      return '<span class="hint" style="color:#c62828">vm = ' + P.vmLo.toFixed(3)
           + ' pu everywhere: this run used the DC power flow, which has no voltage '
           + 'solution. Re-run with [redispatch].power_flow = "AC".</span>';
    return '<span class="hint">Colour = voltage. '
         + '<span style="color:#2166ac">&#9679;</span> ' + (1 - VDEV).toFixed(3)
         + ' &rarr; 1.000 &rarr; ' + (1 + VDEV).toFixed(3)
         + ' <span style="color:#b2182b">&#9679;</span> pu. Observed '
         + P.vmLo.toFixed(3) + '–' + P.vmHi.toFixed(3) + ' pu; band is &plusmn;'
         + (P.vmBand * 100).toFixed(0) + ' %.</span>';
  }
  var mx = BUSMAX[busMode];
  if (busMode === 'net')
    return '<span class="hint">Circle area &prop; |net injection|; '
         + '<span style="color:' + BUSCOL.pos + '">&#9679; surplus</span> / '
         + '<span style="color:' + BUSCOL.neg + '">&#9679; deficit</span>. '
         + 'Largest = ' + fmt(mx) + ' MW.</span>';
  return '<span class="hint">Circle area &prop; ' + MODE_NAME[busMode]
       + '. Largest = ' + fmt(mx) + ' MW.</span>';
}

// ---- control panel --------------------------------------------------------
var Ctl = L.Control.extend({
  options:{position:'topright'},
  onAdd:function(){
    var div = L.DomUtil.create('div', 'gridctl');
    var opts = '', lastGroup = null;
    for (var i = 0; i < P.labels.length; i++){
      if (P.groups[i] !== lastGroup){
        if (lastGroup !== null) opts += '</optgroup>';
        opts += '<optgroup label="' + P.groups[i] + '">';
        lastGroup = P.groups[i];
      }
      opts += '<option value="' + i + '"' + (i === frame ? ' selected' : '') + '>'
            + P.labels[i] + '</option>';
    }
    opts += '</optgroup>';
    div.innerHTML =
      '<b>Frame</b><div class="lbl" id="gridFrameLbl"></div>'
      + '<select id="gridFrameSel">' + opts + '</select>'
      + '<input type="range" id="gridFrameRange" min="0" max="' + (P.labels.length - 1)
      + '" step="1" value="' + frame + '">'
      + '<hr><b>Bus overlay</b>'
      + ['none','gen','dem','net','vm'].map(function(k){
          return '<label><input type="radio" name="gridBusMode" value="' + k + '"'
               + (k === 'none' ? ' checked' : '') + '> '
               + (k === 'none' ? 'none' : MODE_NAME[k]) + '</label>'; }).join('')
      + '<div id="gridBusLegend"></div>'
      + '<hr><b>Lines</b>'
      + '<label><input type="checkbox" id="gridLines" checked> transmission lines</label>'
      + '<label><input type="checkbox" id="gridArrows"> flow-direction arrows (top '
      + ARROW_N + ')</label>';
    L.DomEvent.disableClickPropagation(div);
    L.DomEvent.disableScrollPropagation(div);
    return div;
  }
});
MAP.addControl(new Ctl());

var sel = document.getElementById('gridFrameSel'), rng = document.getElementById('gridFrameRange');
function setFrame(i){ frame = +i; sel.value = i; rng.value = i; MAP.closePopup(); redraw(); }
sel.addEventListener('change', function(){ setFrame(this.value); });
rng.addEventListener('input', function(){ setFrame(this.value); });
Array.prototype.forEach.call(document.getElementsByName('gridBusMode'), function(r){
  r.addEventListener('change', function(){ busMode = this.value; MAP.closePopup(); redraw(); });
});
document.getElementById('gridLines').addEventListener('change', function(){
  if (this.checked) lineGroup.addTo(MAP); else MAP.removeLayer(lineGroup);
});
document.getElementById('gridArrows').addEventListener('change', function(){
  if (this.checked){ rebuildArrows(); arrowGroup.addTo(MAP); } else MAP.removeLayer(arrowGroup);
});
redraw();
});
"""
m.get_root().script.add_child(folium.Element(
    JS.replace('__MAP__', m.get_name()).replace('__DATA__', blob)))
folium.LayerControl(collapsed=False).add_to(m)

out = OUT_MAPS / 'branch_flow_2035.html'
m.save(str(out))
print(f'saved -> {out}  ({out.stat().st_size / 1e6:.1f} MB)')
print(f'opens on: {frame_labels[f0]}   (INITIAL_FRAME at the top of the cell sets this;'
      f' all {NF} frames stay switchable inside the map)')
webbrowser.open(out.resolve().as_uri())
m if SHOW_INLINE else None

## Part III — System adequacy (2035)

Adequacy here = whether generation + imports meet demand **without load shedding** (unserved energy).
This is distinct from the *network* feasibility in Part II — these figures use the energy-balanced
dispatch. **Result: zero load shed in every hour and every market stage.**

*Caveat:* "firm dispatchable" groups Nuclear, Hydro, Gas, Coal, Oil, Biomass, Battery. Reservoir hydro
and batteries are energy-limited, so this slightly overstates sustained firmness.

In [13]:
FIRM = ['Nuclear', 'Hydro', 'Gas', 'Coal', 'Oil', 'Biomass', 'Battery']
snap0    = gen[(gen['date'] == DAYS[0]) & (gen['hour'] == 0)]
firm_cap = snap0[snap0['fuel'].isin(FIRM)]['capacity_mw'].sum()          # firm nameplate [MW]

grp = np.where(gen['fuel'].isin(FIRM), 'firm',
      np.where(gen['fuel'].isin(['Wind', 'Solar']), 'res',
      np.where(gen['fuel'] == 'CrossBorder', 'imp', 'other')))
adq = (gen.assign(grp=grp).pivot_table(index=['date', 'hour'], columns='grp',
       values='dispatch_mw', aggfunc='sum').fillna(0).reset_index())
adq = adq.merge(summary[['date', 'hour', 'total_load_mw', 'load_shed_mw']], on=['date', 'hour'])
adq['reserve'] = firm_cap - adq['firm']

shed_total = adq['load_shed_mw'].clip(lower=0).sum()
imin = adq['reserve'].idxmin()
print('Unserved energy (load shed): %.1f MWh over %d hours  ->  %s'
      % (shed_total, len(adq), 'NONE - system adequate' if shed_total < 0.1 else 'LOAD SHED PRESENT'))
print('Firm dispatchable capacity : %.1f GW  vs peak load %.1f GW'
      % (firm_cap/1000, adq['total_load_mw'].max()/1000))
print('Min firm reserve margin    : %.1f GW  (%s h%02d)'
      % (adq['reserve'].min()/1000, adq.loc[imin, 'date'], int(adq.loc[imin, 'hour'])))

Unserved energy (load shed): 198.5 MWh over 48 hours  ->  LOAD SHED PRESENT
Firm dispatchable capacity : 76.1 GW  vs peak load 63.5 GW
Min firm reserve margin    : 31.4 GW  (2024-12-02 h21)


### Supply vs load — is demand always covered?
Stacked energy supply against load; the dashed red line is total **firm dispatchable capacity**. Load
is served in every hour with firm capacity to spare (load shed = 0).

In [14]:
stack = [('firm', 'firm dispatchable', '#8d6e63'), ('res', 'wind + solar', '#4CAF50'),
         ('imp', 'net imports', '#6C757D')]
fig = make_subplots(rows=1, cols=len(DAYS), shared_yaxes=True,
                    subplot_titles=[DAY_LABEL[d] for d in DAYS])
for ci, d in enumerate(DAYS, start=1):
    a = adq[adq['date'] == d].sort_values('hour')
    for key, lbl, col in stack:
        fig.add_trace(go.Scatter(x=a['hour'], y=a[key].clip(lower=0)/1000, name=lbl, legendgroup=lbl,
                                 showlegend=(ci == 1), stackgroup='s', mode='lines',
                                 line=dict(width=0.5, color=col), fillcolor=col), row=1, col=ci)
    fig.add_trace(go.Scatter(x=a['hour'], y=a['total_load_mw']/1000, name='load', legendgroup='load',
                             showlegend=(ci == 1), mode='lines',
                             line=dict(color='#111', width=2, dash='dot')), row=1, col=ci)
    fig.add_hline(y=firm_cap/1000, line=dict(color='#c62828', width=1.5, dash='dash'),
                  annotation_text='firm capacity' if ci == 1 else None, row=1, col=ci)
    fig.update_xaxes(title_text='hour', row=1, col=ci)
fig.update_yaxes(title_text='GW', row=1, col=1)
fig.update_layout(title='2035 adequacy — supply vs load (load shed = 0 every hour)',
                  height=460, hovermode='x unified')
save_plotly(fig, 'adequacy_supply_2035.html'); fig.show()

### Duration curves vs firm capacity
Load and residual load (= what firm units must serve) sorted high→low. The residual never approaches
the firm-capacity ceiling — the margin is comfortable across all hours.

In [15]:
x = np.arange(1, len(adq) + 1) / len(adq) * 100
fig = go.Figure()
fig.add_scatter(x=x, y=np.sort(adq['total_load_mw'])[::-1]/1000, name='load',
                mode='lines', line=dict(color='#111', width=2))
fig.add_scatter(x=x, y=np.sort(adq['firm'])[::-1]/1000, name='residual load (firm dispatch)',
                mode='lines', line=dict(color='#8d6e63', width=2), fill='tozeroy',
                fillcolor='rgba(141,110,99,0.15)')
fig.add_hline(y=firm_cap/1000, line=dict(color='#c62828', dash='dash'),
              annotation_text=f'firm capacity {firm_cap/1000:.0f} GW')
fig.update_layout(title='2035 load & residual-load duration curves vs firm capacity',
                  xaxis_title='share of hours [%]', yaxis_title='GW', height=440, hovermode='x unified')
save_plotly(fig, 'adequacy_duration_2035.html'); fig.show()

### Firm-capacity margin & winter scarcity
Left: the firm capacity stack against peak and winter-peak demand. Right: hourly cost — winter evenings
spike to ~1 M€/h (expensive gas/oil + CO₂), i.e. **economic** scarcity even though no load is shed.

In [16]:
fig = make_subplots(rows=1, cols=2, column_widths=[0.42, 0.58],
                    subplot_titles=('firm capacity vs peak demand', 'hourly system cost (scarcity)'),
                    specs=[[{'type': 'bar'}, {'type': 'xy'}]])
firm_parts = snap0[snap0['fuel'].isin(FIRM)].groupby('fuel')['capacity_mw'].sum().sort_values(ascending=False)
base = 0.0
for f, v in firm_parts.items():
    if v < 1:
        continue
    fig.add_bar(x=['firm capacity'], y=[v/1000], name=f, marker_color=FUEL_COLORS.get(f, '#999'),
                base=base, row=1, col=1)
    base += v / 1000
peak = adq['total_load_mw'].max() / 1000
wpk  = adq[adq['date'] == DAYS[-1]]['total_load_mw'].max() / 1000
for y, lbl, c in [(peak, 'peak load', '#111'), (wpk, 'winter peak', '#1565c0')]:
    fig.add_hline(y=y, line=dict(color=c, dash='dot'), annotation_text=f'{lbl} {y:.0f} GW', row=1, col=1)
for d in DAYS:
    s = summary[summary['date'] == d].sort_values('hour')
    fig.add_trace(go.Scatter(x=s['hour'], y=s['objective_eur_h']/1000, name=f'cost {DAY_LABEL[d]}',
                             mode='lines+markers'), row=1, col=2)
fig.update_yaxes(title_text='GW', row=1, col=1); fig.update_yaxes(title_text='k€/h', row=1, col=2)
fig.update_xaxes(title_text='hour', row=1, col=2)
fig.update_layout(title='2035 adequacy — firm margin & winter scarcity pricing',
                  height=470, barmode='stack')
save_plotly(fig, 'adequacy_capacity_2035.html'); fig.show()

## Conclusions

1. **Market layer is fine.** DA → ID2 → ID3 → CID → Balancing all clear; 2035 Iberian prices ~28 €/MWh,
   midday solar (~59 GW) exceeds load (~55 GW).
2. **Network layer fails on the 2024 grid.** The AC OPF redispatch is infeasible on all 48 hours at the
   baseline `line_rating_factor = 0.80`. It is **purely thermal** — relaxing generator-bus voltage had
   zero effect; raising line ratings to 4× gives 48/48 with no congestion.
3. **The binding hours are overnight low-load** (~30 GW), where northern hydro+wind must export south —
   not the midday solar peak.
4. **Targeted reinforcement suffices.** System-minimum `line_rating_factor ≈ 2.73`, set by corridor
   **LTGES0690**. Of 2345 branches only ~8 need > 2×, ~50 need > 1.5×; the rest are adequate.
5. **Coupling gap to close:** `empire_scenario.jl` scales fleet/demand/zonal-NTCs to 2035 but leaves
   intra-Spain lines at 2024 ratings. The physically correct fix is to ingest OpenEMPIRE's intra-Spain
   transmission reinforcements into bus-level `rate_a`, mapping the reinforcement list above onto real
   corridor upgrades.
6. **Energy-adequate — zero load shedding.** Firm dispatchable capacity (~58 GW) nearly covers peak
   load (~56 GW); RES is surplus. The winter evening peak is *economically* tight (~1 M€/h, expensive
   gas/oil + CO₂) but no load is ever shed. So the 2035 shortfall is a **network** problem, not a
   **generation-adequacy** one.